# Entraînement final et exploration du modèle BERTopic

Ce notebook recharge les paramètres retenus lors de l'optimisation, entraîne le modèle final et explore la structure thématique du corpus.

In [35]:
import gc
import time
import warnings
from itertools import combinations
from pathlib import Path #pour la gestion des chemins de fichiers
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import ParameterGrid
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.feature_extraction.text import CountVectorizer

from umap import UMAP
from hdbscan import HDBSCAN
from hdbscan.validity import validity_index

from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from sentence_transformers import SentenceTransformer #pour les embeddings de phrase

from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

COL_TEXTE = "phrases_lemm"
COL_ROMAN = "roman"

RANDOM_STATE = 42


warnings.filterwarnings("ignore")

## 1. Chargement du corpus et chargement ou calcul des embeddings

In [36]:
df=pd.read_csv(Path("03_corpus_lematise_128.csv", encoding="utf-8"))

CHEMIN_EMBEDDINGS = Path("data_cache/embeddings_sentence_camembert.npy")

if CHEMIN_EMBEDDINGS.exists():
    print("Chargement des embeddings sauvegardés...")
    embeddings = np.load(CHEMIN_EMBEDDINGS, allow_pickle=False)
else:
    embedding_model = SentenceTransformer(
        "dangvantuan/sentence-camembert-base"
    )

    print("Génération des embeddings sémantiques...")
    embeddings = embedding_model.encode(
        df["texte"].tolist(),
        batch_size=64,
        show_progress_bar=True
    )
    CHEMIN_EMBEDDINGS.parent.mkdir(parents=True, exist_ok=True)
    np.save(CHEMIN_EMBEDDINGS, embeddings)
    print(f"Embeddings sauvegardés dans : {CHEMIN_EMBEDDINGS}")

Chargement des embeddings sauvegardés...


## 2. Préparation des documents pour BERTopic

In [37]:
# Vérification des colonnes
assert COL_TEXTE in df.columns, (
    f"La colonne '{COL_TEXTE}' n'existe pas dans df."
)
assert COL_ROMAN in df.columns, (
    f"La colonne '{COL_ROMAN}' n'existe pas dans df."
)

# Conversion des embeddings
embeddings_array_initial = np.asarray(embeddings)

assert len(df) == len(embeddings_array_initial), (
    "Le nombre de lignes de df ne correspond pas "
    "au nombre d'embeddings."
)

# Masque des documents non vides
masque_documents = (df[COL_TEXTE].notna() & df[COL_TEXTE].astype(str).str.strip().ne(""))

# DataFrame utilisé par BERTopic
df_model = (df.loc[masque_documents].copy().reset_index(drop=True))

# Documents
documents = (df_model[COL_TEXTE].astype(str).tolist())

# Romans associés aux documents
romans = (df_model[COL_ROMAN].astype(str).tolist())

# Embeddings alignés
embeddings_array = embeddings_array_initial[masque_documents.to_numpy()]

# Tokenisation simple pour la cohérence C_v
texts_tokenises = [
    document.split()
    for document in documents
]

# Dictionnaire Gensim
dictionary = Dictionary(texts_tokenises)


print("Nombre de documents :", len(documents))
print("Nombre de romans :", df_model[COL_ROMAN].nunique())
print("Dimensions des embeddings :", embeddings_array.shape)
print("Taille du dictionnaire :", len(dictionary))

Nombre de documents : 29421
Nombre de romans : 31
Dimensions des embeddings : (29421, 768)
Taille du dictionnaire : 19176


## 3. Fonctions d'évaluation finale

In [38]:
def extraire_mots_topics(
    topic_model,
    labels,
    top_n_words=10,
    dictionary=None
):
    """
    Extrait les mots des topics, en excluant le topic -1.

    Si un dictionnaire Gensim est fourni, seuls les mots présents
    dans ce dictionnaire sont conservés.
    """

    labels = np.asarray(labels)

    topic_ids = sorted(
        int(topic_id)
        for topic_id in np.unique(labels)
        if topic_id != -1
    )

    topics_words = []

    for topic_id in topic_ids:

        representation = topic_model.get_topic(topic_id)

        if not representation:
            continue

        words = [
            word
            for word, _ in representation[:top_n_words]
        ]

        if dictionary is not None:
            words = [
                word
                for word in words
                if word in dictionary.token2id
            ]

        # Éviter les topics insuffisamment représentés
        if len(words) >= 2:
            topics_words.append(words)

    return topics_words


def calculer_coherence_cv(
    topic_model,
    labels,
    texts_tokenises,
    dictionary,
    top_n_words=10
):
    """
    Calcule la cohérence C_v des topics BERTopic.
    """

    topics_words = extraire_mots_topics(
        topic_model=topic_model,
        labels=labels,
        top_n_words=top_n_words,
        dictionary=dictionary
    )

    if len(topics_words) < 2:
        return np.nan

    try:
        coherence_model = CoherenceModel(
            topics=topics_words,
            texts=texts_tokenises,
            dictionary=dictionary,
            coherence="c_v",
            topn=top_n_words,
            processes=1
        )

        return float(coherence_model.get_coherence())

    except Exception:
        return np.nan

def calculer_diversite_topics(
    topic_model,
    labels,
    top_n_words=10
):
    """
    Diversité lexicale :
    nombre de termes uniques / nombre total de termes.
    """

    topics_words = extraire_mots_topics(
        topic_model=topic_model,
        labels=labels,
        top_n_words=top_n_words,
        dictionary=None
    )

    tous_les_mots = [
        word
        for topic_words in topics_words
        for word in topic_words
    ]

    if not tous_les_mots:
        return np.nan

    return len(set(tous_les_mots)) / len(tous_les_mots)
    

## 4. Chargement des paramètres et entraînement du modèle final

Les paramètres sélectionnés dans le notebook d'optimisation sont rechargés depuis le fichier JSON.

In [39]:
CHEMIN_PARAMETRES = Path("../data/donnees_annex/meilleurs_parametres_bertopic.json")

with CHEMIN_PARAMETRES.open("r", encoding="utf-8") as f:
    best_params = json.load(f)

best_umap_model = UMAP(
    n_neighbors=best_params["n_neighbors"],
    n_components=best_params["n_components"],
    min_dist=0.0,
    metric="cosine",
    random_state=RANDOM_STATE,
    low_memory=True
)

best_hdbscan_model = HDBSCAN(
    min_cluster_size=best_params["min_cluster_size"],
    min_samples=best_params["min_samples"],
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True,
    core_dist_n_jobs=-1
)

best_vectorizer_model = CountVectorizer(
    min_df=2,
    max_df=0.8,
)

best_ctfidf_model = ClassTfidfTransformer(
    reduce_frequent_words=True,
    bm25_weighting=True
)

best_topic_model = BERTopic(
    language="french",
    hdbscan_model=best_hdbscan_model,
    umap_model=best_umap_model,
    vectorizer_model=best_vectorizer_model,
    ctfidf_model=best_ctfidf_model,
    calculate_probabilities=False,
    nr_topics=None,
    verbose=True
)

best_topics_raw, _ = best_topic_model.fit_transform(
    documents,
    embeddings=embeddings_array
)

best_topics_raw = np.asarray(best_topics_raw)


nombre_topics_bruts = len(
    np.unique(
        best_topics_raw[
            best_topics_raw != -1
        ]
    )
)

taux_outliers_brut = np.mean(best_topics_raw == -1)

print("Nombre naturel de topics :", nombre_topics_bruts)

print( f"Taux brut d'outliers : " f"{taux_outliers_brut:.1%}")

display(best_topic_model.get_topic_info())

2026-08-10 16:06:15,317 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-08-10 16:06:25,008 - BERTopic - Dimensionality - Completed ✓
2026-08-10 16:06:25,009 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-08-10 16:06:27,365 - BERTopic - Cluster - Completed ✓
2026-08-10 16:06:27,368 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-08-10 16:06:27,584 - BERTopic - Representation - Completed ✓


Nombre naturel de topics : 17
Taux brut d'outliers : 72.8%


,Topic,Count,Name,Representation,Representative_Docs
0,-1,21415,-1_abbé_jambe_escalier_verre,"[abbé, jambe, escalier, verre, fièvre, cuisine...",[train roulement foudre respiration juge froid...
1,0,1981,0_chéri_écoute_baiser_tai,"[chéri, écoute, baiser, tai, maman, veu, mécha...",[instant silencieux énergie men époux droit ép...
2,1,1234,1_horizon_herbe_verdure_géant,"[horizon, herbe, verdure, géant, toiture, faça...",[bout grouillement point noir croisée lointain...
3,2,871,2_amant_baiser_étreinte_caresse,"[amant, baiser, étreinte, caresse, passé, épou...",[amant particulier rendez-vous baiser meurtre ...
4,3,827,3_rente_spéculation_achat_gain,"[rente, spéculation, achat, gain, économie, ca...",[premier gain hausse mort illustre vivant mépr...
5,4,592,4_cardinal_abbé_huissier_ministre,"[cardinal, abbé, huissier, ministre, eminence,...",[soir antichambre sourire face rose prélat aim...
6,5,473,5_nation_catholicisme_social_science,"[nation, catholicisme, social, science, religi...",[charité guérison aumône humanité souffrant ef...
7,6,465,6_barricade_prussien_batterie_national,"[barricade, prussien, batterie, national, cano...",[midi horizon entier tonnant feu septième prem...
8,7,244,7_campardon_deberl_dîner_altesse,"[campardon, deberl, dîner, altesse, demoiselle...",[sottise affaire ensemble brave marquis discre...
9,8,213,8_instituteur_jésuite_capucin_curé,"[instituteur, jésuite, capucin, curé, politiqu...",[croyance sincère républicain patriote catholi...


## 5. Réaffectation des documents hors topic

Plusieurs seuils sont comparés avant de retenir le seuil final de réaffectation des outliers.

In [40]:
seuils_outliers = [
    0.0,
    0.10,
    0.20,
    0.30,
    0.40,
    0.50
]

comparaisons_seuils = []
topics_par_seuil = {}

for seuil in seuils_outliers:
    topics_test = best_topic_model.reduce_outliers(
        documents,
        best_topics_raw,
        strategy="embeddings",
        embeddings=embeddings_array,
        threshold=seuil)

    topics_test = np.asarray(topics_test)

    topics_par_seuil[seuil] = topics_test

    masque_assignes = topics_test != -1

    tailles_topics = (pd.Series(topics_test[masque_assignes]).value_counts())

    nombre_assignes = int(masque_assignes.sum())

    nombre_reassignes = int(((best_topics_raw == -1) & (topics_test != -1)).sum())

    comparaisons_seuils.append({
        "threshold": seuil,
        "n_topics": len(tailles_topics),
        "outlier_rate_remaining": np.mean(
            topics_test == -1
        ),
        "n_reassigned": nombre_reassignes,
        "smallest_topic": (
            tailles_topics.min()
            if len(tailles_topics) > 0
            else np.nan
        ),
        "largest_topic": (
            tailles_topics.max()
            if len(tailles_topics) > 0
            else np.nan
        ),
        "largest_topic_share": (
            tailles_topics.max() / nombre_assignes
            if nombre_assignes > 0
            else np.nan
        )
    })


comparaison_seuils_df = pd.DataFrame(comparaisons_seuils)

display(
    comparaison_seuils_df.style.format({
        "threshold": "{:.2f}",
        "outlier_rate_remaining": "{:.1%}",
        "largest_topic_share": "{:.1%}"
    })
)

,threshold,n_topics,outlier_rate_remaining,n_reassigned,smallest_topic,largest_topic,largest_topic_share
0,0.00,17,0.0%,21415,389,3499,11.9%
1,0.10,17,0.0%,21415,389,3499,11.9%
2,0.20,17,0.0%,21415,389,3499,11.9%
3,0.30,17,0.0%,21415,389,3499,11.9%
4,0.40,17,0.0%,21414,389,3499,11.9%
5,0.50,17,0.1%,21396,389,3494,11.9%


In [41]:
SEUIL_OUTLIERS_FINAL = 0.30

topics_finaux = np.asarray(topics_par_seuil[SEUIL_OUTLIERS_FINAL])

print(
    f"Taux d'outliers final : "
    f"{np.mean(topics_finaux == -1):.1%}"
)


print(
    "Nombre final de topics :",
    len(np.unique(topics_finaux[topics_finaux != -1])))

distribution_topics = (
    pd.Series(topics_finaux)
    .value_counts()
    .sort_index()
    .rename_axis("Topic")
    .reset_index(name="Count")
)

display(distribution_topics)

Taux d'outliers final : 0.0%
Nombre final de topics : 17


,Topic,Count
0,0,3499
1,1,2662
2,2,2436
3,3,3242
4,4,2277
5,5,1131
6,6,1358
7,7,3272
8,8,1183
9,9,1547


## 6. Raffinement de la représentation lexicale des topics

In [42]:
stopwords_corpus = ["deberl", "men", "yole","embrass", "revien", "tai", "connai", "quidquid", "trouche", "hattoy", "sai", 'fasse','interrompit',
                    'rauque', 'pan', 'croyez', 'faite'
                    ]

vectorizer_final = CountVectorizer(
    stop_words=stopwords_corpus,
    min_df=2,
    max_df=0.80,
    #ngram_range=(1, 2)
)

ctfidf_final = ClassTfidfTransformer(
    reduce_frequent_words=True,
    bm25_weighting=True
)

best_topic_model.update_topics(
    documents,
    topics=topics_finaux,
    vectorizer_model=vectorizer_final,
    ctfidf_model=ctfidf_final,
    top_n_words=10
)

display(best_topic_model.get_topic_info())

2026-08-10 16:06:28,026 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


,Topic,Count,Name,Representation,Representative_Docs
0,0,3499,0_mignon_pense_infamie_rêv,"[mignon, pense, infamie, rêv, joindre, moi, pl...",[instant silencieux énergie men époux droit ép...
1,1,2662,1_berline_futaie_coulée_gazon,"[berline, futaie, coulée, gazon, buisson, saul...",[bout grouillement point noir croisée lointain...
2,2,2436,2_séparation_nerf_anéantissement_renoncement,"[séparation, nerf, anéantissement, renoncement...",[amant particulier rendez-vous baiser meurtre ...
3,3,3242,3_économie_crédit_loyer_centime,"[économie, crédit, loyer, centime, augmentatio...",[premier gain hausse mort illustre vivant mépr...
4,4,2277,4_eminence_vicaire_commission_demain,"[eminence, vicaire, commission, demain, armate...",[soir antichambre sourire face rose prélat aim...
5,5,1131,5_catholicisme_dogme_démocratie_papauté,"[catholicisme, dogme, démocratie, papauté, civ...",[charité guérison aumône humanité souffrant ef...
6,6,1358,6_barricade_insurgé_division_septième,"[barricade, insurgé, division, septième, artil...",[midi horizon entier tonnant feu septième prem...
7,7,3272,7_peignoir_champagne_invitation_serviette,"[peignoir, champagne, invitation, serviette, c...",[sottise affaire ensemble brave marquis discre...
8,8,1183,8_jésuite_laïque_candidat_socialiste,"[jésuite, laïque, candidat, socialiste, clergé...",[croyance sincère républicain patriote catholi...
9,9,1547,9_séant_insomnie_poêle_respiration,"[séant, insomnie, poêle, respiration, cuvette,...",[ahurissement jambe diable divan trouble somme...


## 7. Évaluation du modèle final

In [43]:
# L'analyseur produit exactement les tokens 
# attendus par le nouveau CountVectorizer.

analyseur_final = (vectorizer_final.build_analyzer())

texts_tokenises_final = [
    analyseur_final(document)
    for document in documents
]


dictionary_final = Dictionary(texts_tokenises_final)


coherence_finale = calculer_coherence_cv(
    topic_model=best_topic_model,
    labels=topics_finaux,
    texts_tokenises=texts_tokenises_final,
    dictionary=dictionary_final,
    top_n_words=10
)


diversite_finale = calculer_diversite_topics(
    topic_model=best_topic_model,
    labels=topics_finaux,
    top_n_words=10
)


print(
    f"Cohérence C_v finale : "
    f"{coherence_finale:.3f}"
)

print(
    f"Diversité finale : "
    f"{diversite_finale:.3f}"
)

Cohérence C_v finale : 0.390
Diversité finale : 0.965


## 8. Visualisations thématiques et temporelles

In [44]:
fig = best_topic_model.visualize_hierarchy()
fig.show()

In [45]:
# Calcul de l'évolution temporelle
topics_over_time = best_topic_model.topics_over_time(
    documents,
    df_model["annee"].tolist(),
    nr_bins=15
)

# Noms plus lisibles

# Création du graphique
fig = best_topic_model.visualize_topics_over_time(
    topics_over_time,
    custom_labels=True,
    normalize_frequency=False,
    title="Travail et milieux populaires",
    width=1500,
    height=700
)

# Personnalisation en français
fig.update_traces(mode="lines+markers")

fig.update_layout(
    template="plotly_white",
    xaxis_title="Période de publication",
    yaxis_title="Nombre de segments",
    legend_title_text=None
)



15it [00:00, 22.02it/s]


## 9. Analyse de la distribution des topics par roman

In [46]:
df_resultats = df_model.copy()

df_resultats["topic"] = topics_finaux

df_resultats["est_outlier"] = (df_resultats["topic"] == -1)

display(df_resultats[[COL_ROMAN, COL_TEXTE, "topic", "est_outlier"]].head())

,roman,phrases_lemm,topic,est_outlier
0,1865 La confession de Claude.,hiver matin frais manteau brouillard saison so...,1,False
1,1865 La confession de Claude.,soir vent porte mur flamme lampe ennui morne g...,1,False
2,1865 La confession de Claude.,chambre bel toile blanc meuble simple luisant ...,0,False
3,1865 La confession de Claude.,lèvre cœur reine laurier songe daignion règle ...,0,False
4,1865 La confession de Claude.,-vou brun rieur moisson vendange épi grappe se...,1,False


In [47]:
table_topics_romans_counts = pd.crosstab(df_resultats[COL_ROMAN],df_resultats["topic"])

display(table_topics_romans_counts)

topic,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
roman,,,,,,,,,,,,,,,,,
1865 La confession de Claude.,194,24,21,3,3,0,1,4,1,11,0,9,27,27,4,3,3
1866 Le voeu d une morte.,46,12,75,15,13,1,2,22,5,16,0,3,7,18,7,39,0
1867 Les mysteres de Marseille.,116,16,54,110,173,9,111,45,51,39,19,7,32,18,29,74,1
1867 Therese Raquin.,55,20,121,25,7,1,2,38,3,51,2,6,73,9,9,44,0
1868 Madeleine Ferat.,134,42,222,16,15,1,0,30,10,45,1,6,54,39,7,64,2
Au Bonheur des dames.,93,96,51,220,47,7,17,236,4,35,5,71,17,19,34,87,1
Fecondite.,257,60,143,186,70,59,22,213,22,57,32,17,53,91,26,114,116
Germinal.,91,204,39,180,44,45,144,103,59,71,17,9,74,9,49,29,3
L argent.,108,57,42,291,92,55,19,72,37,13,24,18,19,23,42,62,9


In [48]:
table_topics_romans_proportions = pd.crosstab(
    df_resultats[COL_ROMAN],
    df_resultats["topic"],
    normalize="index"
)

display(table_topics_romans_proportions.style.format("{:.1%}"))

topic,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
roman,,,,,,,,,,,,,,,,,
1865 La confession de Claude.,57.9%,7.2%,6.3%,0.9%,0.9%,0.0%,0.3%,1.2%,0.3%,3.3%,0.0%,2.7%,8.1%,8.1%,1.2%,0.9%,0.9%
1866 Le voeu d une morte.,16.4%,4.3%,26.7%,5.3%,4.6%,0.4%,0.7%,7.8%,1.8%,5.7%,0.0%,1.1%,2.5%,6.4%,2.5%,13.9%,0.0%
1867 Les mysteres de Marseille.,12.8%,1.8%,6.0%,12.2%,19.1%,1.0%,12.3%,5.0%,5.6%,4.3%,2.1%,0.8%,3.5%,2.0%,3.2%,8.2%,0.1%
1867 Therese Raquin.,11.8%,4.3%,26.0%,5.4%,1.5%,0.2%,0.4%,8.2%,0.6%,10.9%,0.4%,1.3%,15.7%,1.9%,1.9%,9.4%,0.0%
1868 Madeleine Ferat.,19.5%,6.1%,32.3%,2.3%,2.2%,0.1%,0.0%,4.4%,1.5%,6.5%,0.1%,0.9%,7.8%,5.7%,1.0%,9.3%,0.3%
Au Bonheur des dames.,8.9%,9.2%,4.9%,21.2%,4.5%,0.7%,1.6%,22.7%,0.4%,3.4%,0.5%,6.8%,1.6%,1.8%,3.3%,8.4%,0.1%
Fecondite.,16.7%,3.9%,9.3%,12.1%,4.6%,3.8%,1.4%,13.8%,1.4%,3.7%,2.1%,1.1%,3.4%,5.9%,1.7%,7.4%,7.5%
Germinal.,7.8%,17.4%,3.3%,15.4%,3.8%,3.8%,12.3%,8.8%,5.0%,6.1%,1.5%,0.8%,6.3%,0.8%,4.2%,2.5%,0.3%
L argent.,11.0%,5.8%,4.3%,29.6%,9.4%,5.6%,1.9%,7.3%,3.8%,1.3%,2.4%,1.8%,1.9%,2.3%,4.3%,6.3%,0.9%


In [49]:
topics_par_roman = (
    best_topic_model
    .topics_per_class(
        documents,
        classes=romans,
        global_tuning=True
    )
)

display(topics_par_roman)

31it [00:00, 70.68it/s]


,Topic,Words,Frequency,Class
0,0,"dividende, anticipé, obligation, centime, dot",108,L argent.
1,1,"comptant, péristyle, baissier, change, ferré",57,L argent.
2,2,"complexe, salissant, baisse, chagrinaient, obs...",42,L argent.
3,3,"émission, assemblée, actionnaire, liquidation,...",291,L argent.
4,4,"traduction, cote, guichet, calcaire, bol",92,L argent.
...,...,...,...,...
506,12,"égratignait, tulette, aplatissaient, couchât, ...",16,La conquete de Plassans.
507,13,"impasse, purgatoire, plissement, vieille, naine",18,La conquete de Plassans.
508,14,"bréviaire, tanneur, conservateur, moquerie, to...",36,La conquete de Plassans.
509,15,"querelleux, taquin, collégien, cartable, souffler",51,La conquete de Plassans.


## 10. Export des résultats

In [50]:
chemin_sortie = Path("..") /"data" /"4_resultats" /"documents_avec_topics.csv"
chemin_sortie.parent.mkdir(parents=True, exist_ok=True)
df_resultats.to_csv(chemin_sortie, index=False, encoding="utf-8")


chemin_sortie = Path("..") /"data" /"4_resultats" /"topics_par_roman_comptages.csv"
chemin_sortie.parent.mkdir(parents=True, exist_ok=True)
table_topics_romans_counts.to_csv(chemin_sortie, index=True, encoding="utf-8")

chemin_sortie = Path("..") /"data" /"4_resultats" /"topics_par_roman_proportions.csv"
chemin_sortie.parent.mkdir(parents=True, exist_ok=True)
table_topics_romans_proportions.to_csv(chemin_sortie, index=True, encoding="utf-8")

chemin_sortie = Path("..") /"data" /"4_resultats" /"bertopic_topics_per_class.csv"
chemin_sortie.parent.mkdir(parents=True, exist_ok=True)
topics_par_roman.to_csv(chemin_sortie, index=False, encoding="utf-8")

chemin_sortie = Path("..") /"data" /"4_resultats" /"informations_topics.csv"
chemin_sortie.parent.mkdir(parents=True, exist_ok=True)
best_topic_model.get_topic_info().to_csv(chemin_sortie, index=False, encoding="utf-8")


